# FINS 实验流水线

一条 notebook 串起 4 个模块（原 notebook 已 py 化）：

| 步骤       | 模块 | 作用                                                                                           |
|------------|---|------------------------------------------------------------------------------------------------|
| 1 生成配置 | `load.py` | 按拓扑 / 时序 / 负载随机生成 pipeline cfg → `pipeline/*.json`                             |
| 2 采集     | `test.py` | 【fins 测试专用】每份 cfg 起 `bin/client` + `bin/server` 跑 `dur_s` 秒 → trace 复制到 `result/`              |
| 3 标准化   | `std.py` | 【fins 测试专用】 用 JSON 的执行用时在 `execute→complete` 内插 `working` 行 → `result_std/` |
| 4 分析画图 | `plot.py` | 核心甘特、核利用率、生命周期分布                                                               |

> 三个数据目录都在**仓库根**：`pipeline/`（配置）、`result/`（原始 trace）、`result_std/`（标准化 trace）。
> 下面所有相对路径都按仓库根解析，不依赖 notebook 的 cwd。

In [7]:
import importlib

import load
import plot
import std
import test

for m in (load, test, std, plot):
    importlib.reload(m)  # 改了 .py 后重跑本格即可生效

## 1. 生成配置（`load.py`）

5 种拓扑（multihop / fork / join / feedback / mixed，均可混入 acc 的 hist 窗口读）
+ 时序（`ptimed`、`period_divisors`）+ 负载（`u`、`H_ms` 等）随机生成，
文件名 `<kind>_u<u>_m<m>_ms<桶>_s<seed>.json`。

In [ ]:
# 生成一批 cfg（落盘到 <仓库根>/pipeline/）。规模/分布全部由 load.CONFIG 控制：
#   load.CONFIG["topology"] / ["temporal"] / ["workload"]（workload["out_dir"] 相对仓库根）
load.generate_all()

## 2. 采集（`test.py`）

对每份 cfg：起 `client` + 发配置 + 跑 `dur_s` 秒 → 终止 → 把 `tool/temp/tracing.csv`
复制成 `result/<cfg名>.csv`（原件保留，只搬原始 trace，不算指标）。

正式实验要走独占核：`cores="1-6"` 会改用 `sudo tool/client.sh <cores> <workers>`（需要 root）。

In [ ]:
# 三个参数（输入目录 / 结果目录 / temp 目录）相对路径都按仓库根解析
test.test("pipeline/", "result/", "tool/temp/",
          dur_s=10.0, warm_s=2, trials=1)

# 只要一部分：test.test(..., only="feedback", u=70, m=3)
# 独占核（需 root）：test.test(..., cores="1-6")

## 3. 数据标准化（`std.py`）

FINS 自采集的 trace 没有 `working` 行，`execute→complete` 中间无法区分「占核」与「被抢占」。
按 pipeline JSON 里每个节点明确的执行用时（`parameters[0].value`，us），在
`execute + 用时` 处插入 `working` 行 `nX [CPU c, d us]`，并把 `seq` 重排为全局时间序。
**必须同时提供 JSON**，缺 JSON 的文件会被跳过。

接口只吃三个目录：`std.standardize(trace_dir, json_dir, out_dir)`。

In [ ]:
import os

# 只需要三个目录：原始 trace / pipeline JSON / 标准化输出
REPO      = test.repo_root()                    # 仓库根（不依赖 notebook 的 cwd）
TRACE_DIR = os.path.join(REPO, "result")        # 原始 FINS trace（test.py 的输出）
JSON_DIR  = os.path.join(REPO, "pipeline")      # 与 trace 同名的 pipeline JSON
OUT_DIR   = os.path.join(REPO, "result_std")    # 标准化输出 <name>_timeline.csv

# 扫 TRACE_DIR/*.csv，逐份与 JSON_DIR/<同名>.json 配对转换；缺 JSON 的跳过
std.standardize(TRACE_DIR, JSON_DIR, OUT_DIR)

# 等价命令（cwd=仓库根）：python3 tool/std.py result pipeline result_std

In [ ]:
CSV_STD = os.path.join(OUT_DIR, "feedback_u10_m1_ms00_s21100397_timeline.csv")

# 单份细看（只解析不写盘）
CSV  = os.path.join(TRACE_DIR, "feedback_u10_m1_ms00_s21100397.csv")
JSON = os.path.join(JSON_DIR,  "feedback_u10_m1_ms00_s21100397.json")
print(std.load_durations(JSON))  # JSON 里各节点的明确执行用时(us)
df, info = std.convert(CSV, JSON)
df[(df.tid == df.tid.iloc[0]) & (df.seq.between(500, 512))]

## 4. 分析与画图（`plot.py`）

`parse_all(csv)` 一次解析出三份视图（带缓存）：

- `raw`：原始事件（列 `tid,seq,kind,t_us,cpu,tag`）
- `jobs`：每 job 一行，列名与旧 `plot.ipynb` 的 `_parse_hierarchy_data` 完全一致，
  另加 `algo_seg_n / algo_migrated / algo_cores / algo_cpu_us / algo_gap_us`
- `seg`：每段一行（`working` 给出的显式区间），供核心甘特与核利用率使用

> 口径提醒：`algo_exec_span = complete − execute` 是**墙钟**；`algo_cpu_us` 才是**真实占核**
> （各段 `dur` 之和），差额 `algo_gap_us` 是 job 内部被抢占/离核的时间。
> 画「占核多久」请用 `algo_cpu_us` / `seg`。

In [8]:
# PATH = "typical/FINS/feedback_u70_m3_ms05_s20632672_timeline.csv"  # 有迁移的样本
PATH = "typical/CIE_FIFO_IPC/feedback_u70_m3_ms05_s20632672_233736_timeline.csv"

raw, jobs, seg = plot.parse_all(PATH)
print(jobs.groupby("algo").agg(n=("jid", "size"),
                               段数=("algo_seg_n", "mean"),
                               墙钟=("algo_exec_span", "mean"),
                               占核=("algo_cpu_us", "mean"),
                               被抢占=("algo_gap_us", "mean")).round(1))

📊 feedback_u70_m3_ms05_s20632672_233736_timeline.csv: events=6246 jobs=761 segments=1535 迁移 job=13
        n   段数       墙钟       占核     被抢占
algo                                    
n0     76  2.1  34461.8  32590.1  1871.7
n1     76  2.0  14578.9  14577.2     1.7
n2     76  2.0  31304.7  31302.0     2.7
n3     76  2.0  22439.4  22429.9     9.5
n4     76  2.0  11488.4   9973.0  1515.4
n5    153  2.0    206.8    193.6    13.1
n6     76  2.0  52891.2  52866.9    24.3
n7     76  2.1  44936.3  44771.7   164.6
n8     76  2.0   1108.9   1108.5     0.4


In [9]:
# 核心甘特：直接用 working 给出的显式区间，每核一条泳道
plot.draw_core_gantt(seg).show()
# plot.draw_core_gantt(seg, only_migrated=True).show()   # 只看跨核 job
# plot.draw_core_gantt(seg, win=(2000, 4000)).show()     # 只看 2300~2400 ms 窗口

In [ ]:
# 核利用率：Active 按每段真实所在核归属（旧版按 job 的 core_id 归会给迁移线程算错）
plot.draw_core_utilization(raw, seg, jobs).show()

In [ ]:
# 生命周期各段（列名与旧 plot.ipynb 一致，原来的 violin / donut 代码可直接复用）
jobs[["algo", "wake_2_held_lat", "held_2_exec_lat", "algo_exec_span", "exec_2_route_lat",
      "route_2_sleep_lat", "thread_pre_exec_lat", "thread_post_exec_lat", "thread_overhead",
      "algo_seg_n", "algo_migrated", "algo_cpu_us", "algo_gap_us"]].groupby("algo").mean().round(1)

---

### 备注

- 三个数据目录固定在**仓库根**：`pipeline/`（load.py 写）、`result/`（test.py 写）、
  `result_std/`（std.py 写）。三个模块的相对路径都按仓库根解析，notebook 里用
  `test.repo_root()` 拿到根，所以 cwd 是 `tool/` 还是仓库根都一样。
- 原 notebook 已转成同名 `.py`，顶层执行都收在 `if __name__ == "__main__":` 里，
  所以既能直接 `python3 test.py` 跑，也能像上面这样 import 后按需调用。
- 独占核实验需要 root：终端里 `sudo tool/client.sh 1-6 6` 起 worker，
  另开终端 `tool/server.sh pipeline/<cfg>.json` 发配置（`server.sh` 与 `client.sh` 对称）。